In [20]:
import torch
import torch.nn as nn
import yaml
import h5py

import numpy as np

from models.fno import FNO1d
from models.pitt import PhysicsInformedTokenTransformer
from utils import TransformerOperatorDataset

device = 'cuda' if(torch.cuda.is_available()) else 'cpu'

In [21]:
with open("./configs/pitt_config.yaml", 'r') as stream:
        config = yaml.safe_load(stream)

train_args = config['args']
prefix = train_args['flnm'] + "_" + train_args['data_name'].split("_")[0] + "_" + train_args['train_style'] + "_" + \
             train_args['embedding']
train_args['prefix'] = prefix

In [22]:
neural_operator = FNO1d(train_args['num_channels'], train_args['modes'], train_args['width'], train_args['initial_step'], train_args['dropout'])

In [23]:
transformer = PhysicsInformedTokenTransformer(500, train_args['hidden'], train_args['layers'], train_args['heads'],
                                    train_args['num_x'], dropout=train_args['dropout'], neural_operator=neural_operator).to(device=device)

In [24]:
def get_data(f, config):
    test_data = TransformerOperatorDataset(f, config['flnm'],
                            split="test",
                            initial_step=config['initial_step'],
                            reduced_resolution=config['reduced_resolution'],
                            reduced_resolution_t=config['reduced_resolution_t'],
                            reduced_batch=config['reduced_batch'],
                            saved_folder=config['base_path'],
                            return_text=config['return_text'],
                            num_t=config['num_t'],
                            num_x=config['num_x'],
                            sim_time=config['sim_time'],
                            num_samples=config['num_samples'],
                            train_style=config['train_style'],
                            rollout_length=config['rollout_length'],
                            interval=config['interval'],
    )
    test_data.data = test_data.data.to(device)
    test_data.grid = test_data.grid.to(device)

    test_loader = torch.utils.data.DataLoader(test_data, batch_size=config['batch_size'],
                                             num_workers=config['num_workers'], shuffle=False,
                                             generator=torch.Generator(device=device))
    
    return test_loader


In [25]:
def evaluate(test_loader, transformer, loss_fn):
    #src_mask = generate_square_subsequent_mask(640).cuda()
    with torch.no_grad():
        transformer.eval()
        test_loss = 0
        for bn, (x0, y, grid, tokens, t) in enumerate(test_loader):

            y_pred = transformer(grid.to(device=device), tokens.to(device=device), x0.to(device=device), t.to(device=device))

            y = y[...,0].to(device=device)

            # Compute the loss.
            test_loss += loss_fn(y_pred, y).item()
    return test_loss/(bn+1)

In [26]:
loss_list = []
for seed in range(5):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_path = f"1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/model_param_{seed}.pt"
    
    f = h5py.File("{}{}".format(train_args['base_path'], train_args['data_name']), 'r')
    test_loader = get_data(f, train_args)

    loss_fn = nn.MSELoss(reduction='mean')

    transformer.load_state_dict(torch.load(model_path)['model_param'])
    test_value = evaluate(test_loader, transformer, loss_fn)
    print(f'Loss test set seed {seed}:', test_value)
    loss_list.append(test_value)


SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:15<00:00, 760.03it/s]



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1063.73it/s]
/local_scratch/ipykernel_1402280/2834086676.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load(model_pa

Loss test set seed 0: 0.0011334887791197112

SEED: 0
Gathering data...


...

Loss test set seed 3: 0.0011451732679369285

SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:15<00:00, 778.29it/s] 



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1061.95it/s]


Loss test set seed 4: 0.02387520208201827


In [27]:
print(loss_list)

[0.0011334887791197112, 0.0011880607269902496, 0.001268124650604032, 0.0011451732679369285, 0.02387520208201827]


In [28]:
loss_list_end = []
for seed in range(5):
    torch.manual_seed(seed)
    np.random.seed(seed)git
    model_path = f"1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/model_param_end_{seed}.pt"
    
    f = h5py.File("{}{}".format(train_args['base_path'], train_args['data_name']), 'r')
    test_loader = get_data(f, train_args)

    loss_fn = nn.MSELoss(reduction='mean')

    transformer.load_state_dict(torch.load(model_path)['model_param'])
    test_value = evaluate(test_loader, transformer, loss_fn)
    print(f'Loss test set seed {seed}:', test_value)
    loss_list_end.append(test_value)


SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:16<00:00, 729.26it/s] 



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1058.10it/s]
/local_scratch/ipykernel_1402280/1071072472.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load(model_pa

Loss test set seed 0: 0.0011524579672271664

SEED: 0
Gathering data...


...

Loss test set seed 3: 0.001131197443836309

SEED: 0
Gathering data...


100%|██████████| 12000/12000 [00:15<00:00, 771.53it/s] 



NUMBER OF SAMPLES: 12000
Processing data...


12000it [00:11, 1058.57it/s]


Loss test set seed 4: 0.023473857109375457


In [29]:
print(loss_list_end)

[0.0011524579672271664, 0.001188819356104161, 0.0012635978216186483, 0.001131197443836309, 0.023473857109375457]


In [30]:
import csv

step = train_args['initial_step']
interval = train_args['interval']

with open(f'1D_results_time_cont/pitt_fno_Heat_varied_interpolate_novel/test_vals_step{step}_int{interval}_test.csv', mode ='r')as file:
          csvFile = csv.reader(file)
          loss = []
          for line in csvFile:
              line = [float(i) for i in line]
              loss.append(line)

print('test loss', loss[0])
print('best loss', loss[3])

test loss [0.001142827095463872, 0.0011768804707049214, 0.0013237180469428842, 0.0011456737764169798, 0.024010150970772227]
best loss [0.0011425124155605172, 0.0012064803425133467, 0.0012440135686390815, 0.0011223654530229086, 0.024109742921242055]
